In [ ]:

import numpy as np
import math
import matplotlib.pyplot as plt

def wrap_pi(a: float) -> float:
    """Wrap angle to [-pi, pi]."""
    return (a + np.pi) % (2*np.pi) - np.pi

In [ ]:
class PositionPIDController:
    def __init__(self, kp, ki, kd, dt, max_linear_velocity):
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.dt = dt
        self.max_linear_velocity = max_linear_velocity

        # Initialize PID terms
        self.prev_error = 0.0
        self.integral = 0.0

    def compute_linear_velocity(self, target_position, current_position):
        # Calculate the distance to the target
        distance_error = np.linalg.norm(np.array(target_position) - np.array(current_position))

        # Calculate PID terms
        error = distance_error
        self.integral += error * self.dt
        derivative = (error - self.prev_error) / self.dt
        self.prev_error = error

        # Compute control output
        linear_velocity = self.kp * error + self.ki * self.integral + self.kd * derivative

        # Apply maximum linear velocity limit
        if linear_velocity > self.max_linear_velocity:
            linear_velocity = self.max_linear_velocity
        elif linear_velocity < -self.max_linear_velocity:
            linear_velocity = -self.max_linear_velocity

        return linear_velocity


In [ ]:
class AnglePIDController:
    def __init__(self, kp, ki, kd, dt, max_angular_velocity):
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.dt = dt
        self.max_angular_velocity = max_angular_velocity

        # Initialize PID terms
        self.prev_error = 0.0
        self.integral = 0.0

    def compute_angular_velocity(self, goal_angle, curr_angle):
        # Calculate the angle difference
        angle_diff = goal_angle - curr_angle
        # Normalize angle_diff to the range [-pi, pi]
        angle_diff = (angle_diff + np.pi) % (2 * np.pi) - np.pi

        # Calculate PID terms
        error = angle_diff
        self.integral += error * self.dt
        derivative = (error - self.prev_error) / self.dt
        self.prev_error = error

        # Compute control output
        angular_velocity = self.kp * error + self.ki * self.integral + self.kd * derivative

        # Apply maximum angular velocity limit
        if angular_velocity > self.max_angular_velocity:
            angular_velocity = self.max_angular_velocity
        elif angular_velocity < -self.max_angular_velocity:
            angular_velocity = -self.max_angular_velocity

        return angular_velocity

In [ ]:
# Global timing
dt = 0.02
T  = 12.0
N  = int(T/dt)

# Position PID gains/limits
pos_kp, pos_ki, pos_kd = 1.2, 0.0, 0.25
v_max = 1.2

# Angle PID gains/limits
ang_kp, ang_ki, ang_kd = 2.5, 0.0, 0.2
w_max = 2.0

In [ ]:
target_pos_1d = np.array([0.0])
start_pos_1d  = np.array([5.0])

pos_pid = PositionPIDController(pos_kp, pos_ki, pos_kd, dt, v_max)

x = start_pos_1d.copy()
x_hist, e_hist, v_hist, t = [], [], [], []

for i in range(N):
    v = pos_pid.compute_linear_velocity(target_pos_1d, x)
    # Discrete update toward the target (point mass with direct velocity command)
    x = x - v * dt * np.sign(x - target_pos_1d)
    x_hist.append(float(x))
    e_hist.append(float(np.abs(x - target_pos_1d)))
    v_hist.append(float(v))
    t.append(i*dt)

plt.figure()
plt.plot(t, x_hist, label="Position (m)")
plt.axhline(float(target_pos_1d), linestyle="--", label="Target")
plt.xlabel("Time (s)"); plt.ylabel("Position (m)")
plt.title("1D Position vs Time (Position PID)"); plt.legend(); plt.grid(True)
plt.show()

plt.figure()
plt.plot(t, e_hist, label="Distance Error (m)")
plt.xlabel("Time (s)"); plt.ylabel("Error (m)")
plt.title("1D Distance Error vs Time"); plt.legend(); plt.grid(True)
plt.show()

plt.figure()
plt.plot(t, v_hist, label="Linear Velocity Command (m/s)")
plt.xlabel("Time (s)"); plt.ylabel("Velocity (m/s)")
plt.title("Linear Velocity vs Time"); plt.legend(); plt.grid(True)
plt.show()


In [ ]:
goal_heading  = 0.0
start_heading = np.deg2rad(120)

ang_pid = AnglePIDController(ang_kp, ang_ki, ang_kd, dt, w_max)

theta = start_heading
theta_hist, err_hist, w_hist, t2 = [], [], [], []

for i in range(N):
    w = ang_pid.compute_angular_velocity(goal_heading, theta)
    theta = wrap_pi(theta + w*dt)
    theta_hist.append(theta)
    err_hist.append(wrap_pi(goal_heading - theta))
    w_hist.append(w)
    t2.append(i*dt)

plt.figure()
plt.plot(t2, theta_hist, label="Heading (rad)")
plt.axhline(goal_heading, linestyle="--", label="Goal")
plt.xlabel("Time (s)"); plt.ylabel("Angle (rad)")
plt.title("Heading vs Time (Angle PID)"); plt.legend(); plt.grid(True)
plt.show()

plt.figure()
plt.plot(t2, err_hist, label="Heading Error (rad)")
plt.xlabel("Time (s)"); plt.ylabel("Error (rad)")
plt.title("Heading Error vs Time"); plt.legend(); plt.grid(True)
plt.show()

plt.figure()
plt.plot(t2, w_hist, label="Angular Velocity Command (rad/s)")
plt.xlabel("Time (s)"); plt.ylabel("Angular Velocity (rad/s)")
plt.title("Angular Velocity vs Time"); plt.legend(); plt.grid(True)
plt.show()

In [ ]:
# Robot kinematics: x_dot = v*cos(theta), y_dot = v*sin(theta), theta_dot = w

goal_xy = np.array([5.0, 5.0])
start_xy = np.array([-4.0, -2.0])
start_theta = np.deg2rad(-100)

pos_pid2 = PositionPIDController(pos_kp, pos_ki, pos_kd, dt, v_max)
ang_pid2 = AnglePIDController(ang_kp, ang_ki, ang_kd, dt, w_max)

xy = start_xy.astype(float).copy()
th = float(start_theta)

traj = [xy.copy()]
d_errs, a_errs, vs, ws, t3 = [], [], [], [], []

for i in range(N):
    delta = goal_xy - xy
    dist = float(np.linalg.norm(delta))
    goal_ang = math.atan2(delta[1], delta[0])

    v_cmd = pos_pid2.compute_linear_velocity(goal_xy, xy)
    w_cmd = ang_pid2.compute_angular_velocity(goal_ang, th)

    # Optional: reduce v when heading error is large (uncomment to try)
    # v_cmd *= max(0.0, 1.0 - abs(wrap_pi(goal_ang - th))/np.pi)

    xy = xy + np.array([math.cos(th), math.sin(th)]) * v_cmd * dt
    th = wrap_pi(th + w_cmd * dt)
    traj.append(xy.copy())

    d_errs.append(dist)
    a_errs.append(wrap_pi(goal_ang - th))
    vs.append(v_cmd)
    ws.append(w_cmd)
    t3.append(i*dt)

    if dist < 0.05:
        break

traj = np.array(traj)

plt.figure()
plt.plot(traj[:,0], traj[:,1], label="Path")
plt.scatter([start_xy[0]], [start_xy[1]], marker="o", label="Start")
plt.scatter([goal_xy[0]], [goal_xy[1]], marker="x", label="Goal")
plt.axis("equal")
plt.xlabel("X (m)"); plt.ylabel("Y (m)")
plt.title("2D Path with Position+Angle PID (Unicycle)")
plt.legend(); plt.grid(True)
plt.show()

plt.figure()
plt.plot(t3, d_errs, label="Distance Error (m)")
plt.xlabel("Time (s)"); plt.ylabel("Error (m)")
plt.title("2D Navigation: Distance Error vs Time"); plt.legend(); plt.grid(True)
plt.show()

plt.figure()
plt.plot(t3, a_errs, label="Heading Error (rad)")
plt.xlabel("Time (s)"); plt.ylabel("Error (rad)")
plt.title("2D Navigation: Heading Error vs Time"); plt.legend(); plt.grid(True)
plt.show()

plt.figure()
plt.plot(t3, vs, label="Linear Velocity (m/s)")
plt.xlabel("Time (s)"); plt.ylabel("Velocity (m/s)")
plt.title("2D Navigation: Linear Velocity vs Time"); plt.legend(); plt.grid(True)
plt.show()

plt.figure()
plt.plot(t3, ws, label="Angular Velocity (rad/s)")
plt.xlabel("Time (s)"); plt.ylabel("Angular Velocity (rad/s)")
plt.title("2D Navigation: Angular Velocity vs Time"); plt.legend(); plt.grid(True)
plt.show()